In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import warnings
import csv
warnings.filterwarnings("ignore", message=".*observed=False.*")

In [2]:
notes_path = "discharge.csv.gz"

#### Load output from create_basline_data.csv

In [3]:
stroke_df_path = "complete_df_nn.csv"

In [4]:
stroke_df = pd.read_csv(stroke_df_path)
stroke_patient_ids = stroke_df["subject_id"].tolist()
print(len(stroke_df), len(stroke_patient_ids))

26948 26948


#### Load file with clinical notes

In [5]:
notes = pd.read_csv(notes_path, parse_dates=["charttime","storetime"], compression='gzip')
notes.head()

,note_id,subject_id,hadm_id,note_type,note_seq,charttime,storetime,text
0,10000032-DS-21,10000032,22595853,DS,21,2180-05-07,2180-05-09 15:26:00,\nName: ___ Unit No: _...
1,10000032-DS-22,10000032,22841357,DS,22,2180-06-27,2180-07-01 10:15:00,\nName: ___ Unit No: _...
2,10000032-DS-23,10000032,29079034,DS,23,2180-07-25,2180-07-25 21:42:00,\nName: ___ Unit No: _...
3,10000032-DS-24,10000032,25742920,DS,24,2180-08-07,2180-08-10 05:43:00,\nName: ___ Unit No: _...
4,10000084-DS-17,10000084,23052089,DS,17,2160-11-25,2160-11-25 15:09:00,\nName: ___ Unit No: __...


In [6]:
stroke_df.head()

,Unnamed: 0,subject_id,icd_code,gender,anchor_age,dod,mortality,hadm_id,admittime,dischtime,deathtime,admission_type,race,marital_status,number_of_readmissions,icd_version,age_group
0,0,10000980,"25000,25040,25060,2724,27800,2851,28521,28749,...",F,73,2193-08-26,1,29659838,2191-07-16 14:21:00,2191-07-19 13:03:00,NaN,EW EMER.,BLACK/AFRICAN AMERICAN,MARRIED,5,10,65–74
1,1,10001877,"2252,25000,2724,4019,412,42731,4280,42833,4341...",M,89,NaN,0,25679292,2149-05-21 15:53:00,2149-05-27 13:35:00,NaN,EW EMER.,WHITE,MARRIED,0,9,85+
2,2,10002155,"1628,2469,2720,2724,2761,2767,2809,2851,28522,...",F,80,2131-03-10,1,28439444,2128-07-29 17:01:00,2128-07-31 18:00:00,NaN,EW EMER.,WHITE,MARRIED,0,9,75–84
3,3,10002430,"C439,C7951,C799,D638,D696,E7849,E785,E871,I071...",M,86,2130-01-11,1,24648311,2129-04-29 12:24:00,2129-05-02 16:37:00,NaN,DIRECT EMER.,WHITE,WIDOWED,0,10,85+
4,4,10003019,"00845,0389,135,20190,20193,20280,2724,2753,275...",M,69,NaN,0,21223482,2175-10-31 22:26:00,2175-11-02 15:30:00,NaN,EW EMER.,WHITE,MARRIED,0,9,65–74


In [7]:
stroke_df["age_group"].unique()

array(['65–74', '85+', '75–84', '45–64', '18–44'], dtype=object)

#### Confirming that hadm_id is a unique identifier for the stroke notes

In [8]:
len(notes["hadm_id"].unique())

331793

In [9]:
len(notes)

331793

In [10]:
notes["text"].isnull().sum()

np.int64(0)

In [11]:
len(stroke_df)

26948

#### Get notes for each patient based on subject_id and hadm_id

In [12]:
complete_stroke_data = notes.merge(stroke_df, on=["subject_id", "hadm_id"],how="inner")
print(len(complete_stroke_data))

17890


In [13]:
complete_stroke_data.head()

,note_id,subject_id,hadm_id,note_type,note_seq,charttime,storetime,text,Unnamed: 0,icd_code,...,mortality,admittime,dischtime,deathtime,admission_type,race,marital_status,number_of_readmissions,icd_version,age_group
0,10000980-DS-25,10000980,29659838,DS,25,2191-07-19,2191-07-22 09:37:00,\nName: ___ Unit No: ___\n \nAdmi...,0,"25000,25040,25060,2724,27800,2851,28521,28749,...",...,1,2191-07-16 14:21:00,2191-07-19 13:03:00,NaN,EW EMER.,BLACK/AFRICAN AMERICAN,MARRIED,5,10,65–74
1,10001877-DS-19,10001877,25679292,DS,19,2149-05-27,2149-05-27 11:29:00,\nName: ___ ___ No: ___\n \...,1,"2252,25000,2724,4019,412,42731,4280,42833,4341...",...,0,2149-05-21 15:53:00,2149-05-27 13:35:00,NaN,EW EMER.,WHITE,MARRIED,0,9,85+
2,10002430-DS-6,10002430,24648311,DS,6,2129-05-02,2129-05-02 18:31:00,\nName: ___ Unit No: ___\n \n...,3,"C439,C7951,C799,D638,D696,E7849,E785,E871,I071...",...,1,2129-04-29 12:24:00,2129-05-02 16:37:00,NaN,DIRECT EMER.,WHITE,WIDOWED,0,10,85+
3,10003019-DS-23,10003019,21223482,DS,23,2175-11-02,2175-11-02 22:33:00,\nName: ___. Unit No: ___\n \...,4,"00845,0389,135,20190,20193,20280,2724,2753,275...",...,0,2175-10-31 22:26:00,2175-11-02 15:30:00,NaN,EW EMER.,WHITE,MARRIED,0,9,65–74
4,10003299-DS-10,10003299,27373340,DS,10,2183-08-01,2183-08-01 18:44:00,\nName: ___ Unit No: ___\n...,5,"2724,3051,4011,4019,41401,43411,43491,56210,56...",...,1,2183-07-23 20:41:00,2183-08-01 18:30:00,NaN,URGENT,BLACK/AFRICAN AMERICAN,WIDOWED,9,10,65–74


In [14]:
stroke_df.columns

Index(['Unnamed: 0', 'subject_id', 'icd_code', 'gender', 'anchor_age', 'dod',
       'mortality', 'hadm_id', 'admittime', 'dischtime', 'deathtime',
       'admission_type', 'race', 'marital_status', 'number_of_readmissions',
       'icd_version', 'age_group'],
      dtype='object')

##### Some patients don't have clinical notes for the specific stroke-related hospital visit

In [15]:
complete_stroke_data.isnull().sum()

,0
note_id,0
subject_id,0
hadm_id,0
note_type,0
note_seq,0
charttime,0
storetime,0
text,0
Unnamed: 0,0
icd_code,0


##### With all the preprocessing done, better to check that the mortality column was not compromised

In [16]:
check_mortality = (
    ((complete_stroke_data["dod"].isna()) & (complete_stroke_data["mortality"] != 0)) |
    ((~complete_stroke_data["dod"].isna()) & (complete_stroke_data["mortality"] != 1))
)
print("Number of inconsistent rows:", check_mortality.sum())


Number of inconsistent rows: 0


In [17]:
print("Length of the full dataset:", len(complete_stroke_data))

Length of the full dataset: 17890


In [18]:
complete_stroke_data.drop_duplicates(subset=["subject_id", "hadm_id"],keep="first")
len(complete_stroke_data)

17890

#### Format notes properly to fix intances that extend to more than one row

In [19]:
complete_stroke_data["text"] = complete_stroke_data["text"].replace({r'\r\n': ' ', r'\n': ' '}, regex=True)
complete_stroke_data.to_csv("complete_clean_data.csv", index=False, quoting=csv.QUOTE_ALL, escapechar="\\", encoding="utf-8")

In [20]:
df_complete = pd.read_csv("complete_clean_data.csv", quoting=csv.QUOTE_ALL, escapechar="\\", encoding="utf-8")
#assert complete_stroke_data.equals(df_complete)
#print("Data properly encoded!")

In [21]:
complete_stroke_data.columns

Index(['note_id', 'subject_id', 'hadm_id', 'note_type', 'note_seq',
       'charttime', 'storetime', 'text', 'Unnamed: 0', 'icd_code', 'gender',
       'anchor_age', 'dod', 'mortality', 'admittime', 'dischtime', 'deathtime',
       'admission_type', 'race', 'marital_status', 'number_of_readmissions',
       'icd_version', 'age_group'],
      dtype='object')

In [22]:
df_complete.head()

,note_id,subject_id,hadm_id,note_type,note_seq,charttime,storetime,text,Unnamed: 0,icd_code,...,mortality,admittime,dischtime,deathtime,admission_type,race,marital_status,number_of_readmissions,icd_version,age_group
0,10000980-DS-25,10000980,29659838,DS,25,2191-07-19,2191-07-22 09:37:00,Name: ___ Unit No: ___ Admissi...,0,"25000,25040,25060,2724,27800,2851,28521,28749,...",...,1,2191-07-16 14:21:00,2191-07-19 13:03:00,NaN,EW EMER.,BLACK/AFRICAN AMERICAN,MARRIED,5,10,65–74
1,10001877-DS-19,10001877,25679292,DS,19,2149-05-27,2149-05-27 11:29:00,Name: ___ ___ No: ___ Ad...,1,"2252,25000,2724,4019,412,42731,4280,42833,4341...",...,0,2149-05-21 15:53:00,2149-05-27 13:35:00,NaN,EW EMER.,WHITE,MARRIED,0,9,85+
2,10002430-DS-6,10002430,24648311,DS,6,2129-05-02,2129-05-02 18:31:00,Name: ___ Unit No: ___ Adm...,3,"C439,C7951,C799,D638,D696,E7849,E785,E871,I071...",...,1,2129-04-29 12:24:00,2129-05-02 16:37:00,NaN,DIRECT EMER.,WHITE,WIDOWED,0,10,85+
3,10003019-DS-23,10003019,21223482,DS,23,2175-11-02,2175-11-02 22:33:00,Name: ___. Unit No: ___ Ad...,4,"00845,0389,135,20190,20193,20280,2724,2753,275...",...,0,2175-10-31 22:26:00,2175-11-02 15:30:00,NaN,EW EMER.,WHITE,MARRIED,0,9,65–74
4,10003299-DS-10,10003299,27373340,DS,10,2183-08-01,2183-08-01 18:44:00,Name: ___ Unit No: ___ ...,5,"2724,3051,4011,4019,41401,43411,43491,56210,56...",...,1,2183-07-23 20:41:00,2183-08-01 18:30:00,NaN,URGENT,BLACK/AFRICAN AMERICAN,WIDOWED,9,10,65–74


In [24]:
len(df_complete)

17890